In [ ]:
from pathlib import Path

import altair as alt
import polars as pl

In [ ]:
geotab_dir = Path(
    (
        r"O:\Active Studies\Bayview Freight\200_Project_Development_Tasks"
        "\Task 2 - Existing Conditions\geotab-OD"
    )
)
simwrapper_output_dir = geotab_dir / "simwrapper"
od_dir = geotab_dir / "OD-csv_responses-EXCLUDENapaSolano"

In [ ]:
geogs_in_bayview = [
    "230.01",
    "230.03",
    "231.02",
    "231.03",
    "232",
    "233",
    "234",
    "610",
    "612",
    "9806",
    "9809",
]
geogs_ex_bayview_dict = {
    "East Bay": "East Bay",
    "North Bay": "North Bay",
    "Outside of NorCal": "ex Bay Area",
    "San Francisco": "rest of SF",
    "South Bay": "Peninsula & South Bay",
}
geogs_ex_bayview = list(geogs_ex_bayview_dict.values())
geogs = geogs_in_bayview + geogs_ex_bayview

quarters = ["Q1", "Q2", "Q3", "Q4"]
time_periods = ["EA", "AM", "MD", "PM", "EV", "ON"]

In [ ]:
def get_filepath(time_period, quarter):
    glob_pattern = f"{time_period}_{quarter}_response_its_*.csv"
    matches = list(od_dir.glob(glob_pattern))
    if len(matches) != 1:
        raise ValueError(
            (
                f"{len(matches)} files matched "
                f"for time period {time_period} and quarter {quarter}."
            )
        )
    return matches[0]


def long_df_to_simwrapper(long_df):
    return long_df.pivot(
        on="time_period",
        index=["OriginZoneDescription", "DestinationZoneDescription"],
        values="hourly_journeys_expanded",
    ).select(
        pl.col("OriginZoneDescription").alias("origin"),
        pl.col("DestinationZoneDescription").alias("destination"),
        *time_periods,
    )

In [ ]:
time_period_df = pl.read_csv(
    geotab_dir / "geotab-time_period_expansion_factor-Bayview.csv"
)

dfs = [
    pl.read_csv(get_filepath(tp, q))
    .with_columns(
        pl.col("OriginZoneDescription", "DestinationZoneDescription").replace(
            geogs_ex_bayview_dict
        )
    )
    .remove(  # BAYVIEW PROJECT-SPECIFIC
        # since this dataset only captures trips with O or D in Bayview and
        # excludes pass-through trips, delete all ex-Bayview to ex-Bayview rows
        pl.col("OriginZoneDescription").is_in(geogs_ex_bayview)
        & pl.col("DestinationZoneDescription").is_in(geogs_ex_bayview)
    )
    .with_columns(
        # 2 cols have numbers with ',' so need to cast them explicitly to float
        pl.col("DailyJourneyCountAvg", "DurationAtDestinationStdev")
        # HOTFIX if all values in col < 1000, it would already be a float,
        # cast as str first to allow consistent handling
        .cast(pl.Utf8)
        .str.replace_all(",", "")
        .cast(pl.Float64)
    )
    .with_columns(time_period=pl.lit(tp), quarter=pl.lit(q))
    for tp in time_periods
    for q in quarters
]
quarterly_df = (
    pl.concat(dfs, how="diagonal", rechunk=True)
    .join(
        time_period_df,
        # TimeFrom/TimeTo are not in this batch of raw data, so use time_period directly
        on="time_period",  # on=["TimeFrom", "TimeTo"],
    )
    .with_columns(
        hourly_journey_count=pl.col("DailyJourneyCountAvg")
        / pl.col("hours_in_time_period")
    )
    .with_columns(
        hourly_journeys_expanded=pl.col("hourly_journey_count")
        * pl.col("expansion_factor"),
    )
)

In [ ]:
# plot OD matrices as heat map, code NOT updated yet to work directly with long form df
# fig, axs = plt.subplots(
#     len(time_periods),
#     len(quarters),
#     figsize=(40, 50),  # , sharex=True, sharey=True
# )
# for i_tp, tp in enumerate(time_periods):
#     for i_q, q in enumerate(quarters):
#         ax = axs[i_tp][i_q]
#         ax.set_title(f"{tp} {q}")
#         sns.heatmap(
#             dfs[i_tp][i_q].drop("geography"),
#             ax=ax,
#             cmap="viridis",
#             xticklabels=geogs,
#             yticklabels=geogs,
#             annot=True,
#         )
# plt.savefig(od_dir / "bayview-od.png")
# plt.show()

In [ ]:
# plot OD matrices as heat map, code NOT updated yet to work directly with long form df
# with vmax set to the max value outside of 9809 (too many trips in 9809)
# fig, axs = plt.subplots(
#     len(time_periods),
#     len(quarters),
#     figsize=(40, 50),  # , sharex=True, sharey=True
# )
# for i_tp, tp in enumerate(time_periods):
#     for i_q, q in enumerate(quarters):
#         ax = axs[i_tp][i_q]
#         ax.set_title(f"{tp} {q}")
#         vmax = (
#             dfs[i_tp][i_q]
#             .filter(pl.col("geography") != "9809")
#             .select(pl.exclude("9809", "geography"))
#             .max()
#             .max_horizontal()
#             .item()
#         )
#         sns.heatmap(
#             dfs[i_tp][i_q].drop("geography"),
#             ax=ax,
#             vmax=vmax,
#             cmap="viridis",
#             xticklabels=geogs,
#             yticklabels=geogs,
#             annot=True,
#         )
# plt.savefig(od_dir / "bayview-od-vmax_ex9809.png")
# plt.show()

In [ ]:
# plot OD matrices as heat map, code NOT updated yet to work directly with long form df
# with vmax set to the max value outside of 9809 and the diagonal
# fig, axs = plt.subplots(
#     len(time_periods),
#     len(quarters),
#     figsize=(40, 50),  # , sharex=True, sharey=True
# )
# for i_tp, tp in enumerate(time_periods):
#     for i_q, q in enumerate(quarters):
#         ax = axs[i_tp][i_q]
#         ax.set_title(f"{tp} {q}")
#         vmax = (
#             dfs[i_tp][i_q]
#             # set diagonal to null
#             .with_columns(
#                 pl.when(pl.col("geography") == pl.lit(c))
#                 .then(None)
#                 .otherwise(pl.col(c))
#                 .alias(c)
#                 for c in geogs
#             )
#             # don't consider 9809 trips for vmax
#             .filter(pl.col("geography") != "9809")
#             .select(pl.exclude("9809", "geography"))
#             # get max
#             .max()
#             .max_horizontal()
#             .item()
#         )
#         sns.heatmap(
#             dfs[i_tp][i_q].drop("geography"),
#             ax=ax,
#             vmax=vmax,
#             cmap="viridis",
#             xticklabels=geogs,
#             yticklabels=geogs,
#             annot=True,
#         )
# plt.savefig(od_dir / "bayview-od-vmax_ex9809_exdiagonal.png")
# plt.show()

In [ ]:
quarterly_df = quarterly_df.with_columns(
    # number_of_days_averaged can be different even for the same time_period and quarter
    # since Geotab calculates DailyJourneyCountAvg using the number of days that passed
    # the confidentiality filter, treating the number of journeys on days not passing
    # the confidentiality filter as N/A instead of 0.
    number_of_days_averaged=(
        (
            pl.col("JourneyCount") / pl.col("DailyJourneyCountAvg")
        ).round()  # DailyJourneyCountAvg only has 2 decimal places; fix rounding errors
    )
)
annual_df = (
    quarterly_df
    # group all 4 quarters together to get annual average
    .group_by(
        "time_period",
        "OriginZoneDescription",
        "DestinationZoneDescription",
        "hours_in_time_period",
        "expansion_factor",
    )
    .agg(
        JourneyCount=pl.col("JourneyCount").sum(),
        number_of_days_averaged=pl.col("number_of_days_averaged").sum(),
    )
    .with_columns(
        DailyJourneyCountAvg=pl.when(pl.col("number_of_days_averaged") == 0)
        .then(0)
        .otherwise(pl.col("JourneyCount") / pl.col("number_of_days_averaged"))
    )
    .with_columns(
        hourly_journey_count=pl.col("DailyJourneyCountAvg")
        / pl.col("hours_in_time_period")
    )
    .with_columns(
        hourly_journeys_expanded=pl.col("hourly_journey_count")
        * pl.col("expansion_factor"),
    )
)

In [ ]:
alt.data_transformers.disable_max_rows()
alt.Chart(quarterly_df).mark_bar().encode(
    alt.X(
        "number_of_days_averaged:Q",
        bin=alt.Bin(
            maxbins=quarterly_df.select(pl.col("number_of_days_averaged").max()).item()
        ),
    ),
    y="count()",
)

In [ ]:
alt.Chart(annual_df.filter(pl.col("number_of_days_averaged") > 0)).mark_bar().encode(
    alt.X(
        "number_of_days_averaged:Q",
        bin=alt.Bin(maxbins=50),
    ),
    y="count()",
)

In [ ]:
# get summaries of trips within and to/from Bayview
# (missing pass through trips since it's a separate Geotab query)
tofromwithin_journeys = (
    annual_df.with_columns(
        origin_area=pl.when(pl.col("OriginZoneDescription").is_in(geogs_in_bayview))
        .then(pl.lit("Bayview"))
        .when(pl.col("OriginZoneDescription").is_in(geogs_ex_bayview))
        .then(pl.lit("ex Bayview"))
        .otherwise(pl.lit("?")),  # TODO should catch this error instead
        destination_area=pl.when(
            pl.col("DestinationZoneDescription").is_in(geogs_in_bayview)
        )
        .then(pl.lit("Bayview"))
        .when(pl.col("DestinationZoneDescription").is_in(geogs_ex_bayview))
        .then(pl.lit("ex Bayview"))
        .otherwise(pl.lit("?")),  # TODO should catch this error instead
    )
    .group_by("origin_area", "destination_area", "time_period")
    .agg(pl.sum("hourly_journeys_expanded"))
    .sort(["time_period", "origin_area", "destination_area"])
)
tofromwithin_journeys.write_csv(simwrapper_output_dir / "bayview-tofromwithin.csv")

In [ ]:
annual_simwrapper_df = long_df_to_simwrapper(annual_df)
quarterly_simwrapper_df = (
    quarterly_df.pivot(
        on=["time_period", "quarter"],
        index=["OriginZoneDescription", "DestinationZoneDescription"],
        values="hourly_journeys_expanded",
    )
    .rename(
        {"OriginZoneDescription": "origin", "DestinationZoneDescription": "destination"}
    )
    .fill_null(0)
)

In [ ]:
intra_bayview_filter = pl.col("origin").is_in(geogs_in_bayview) & pl.col(
    "destination"
).is_in(geogs_in_bayview)

In [ ]:
quarterly_simwrapper_df.write_csv(simwrapper_output_dir / "bayview-od-quarterly.csv")
annual_simwrapper_df.write_csv(simwrapper_output_dir / "bayview-od-annual.csv")
quarterly_simwrapper_df.filter(intra_bayview_filter).write_csv(
    simwrapper_output_dir / "bayview-od-quarterly-intra_bayview.csv"
)
annual_simwrapper_df.filter(intra_bayview_filter).write_csv(
    simwrapper_output_dir / "bayview-od-annual-intra_bayview.csv"
)